In [ ]:
import os

In [ ]:
ld_library_path = os.environ.get("LD_LIBRARY_PATH", "") # Get current LD_LIBRARY_PATH (if set)
conda_lib_path = 'data/conda/envs/swift/lib:/lib/x86_64-linux-gnu'
new_ld_library_path = f"{conda_lib_path}:{ld_library_path}"
os.environ["LD_LIBRARY_PATH"] = new_ld_library_path # Set it

# model

In [ ]:
import sys
sys.path.append('workplace/RA/repo/Qwen2.5-VL/qwen-vl-utils/src/')

from qwen_vl_utils import process_vision_info

In [ ]:
import torch

from transformers import Qwen2_5_VLForConditionalGeneration, AutoTokenizer, AutoProcessor

In [ ]:
# We recommend enabling flash_attention_2 for better acceleration and memory saving, especially in multi-image and video scenarios.
model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    "Qwen/Qwen2.5-VL-7B-Instruct",
    torch_dtype=torch.float16,
    attn_implementation="flash_attention_2",
    device_map="auto",
)

# default processer
processor = AutoProcessor.from_pretrained("Qwen/Qwen2.5-VL-7B-Instruct")

# The default range for the number of visual tokens per image in the model is 4-16384.
# You can set min_pixels and max_pixels according to your needs, such as a token range of 256-1280, to balance performance and cost.
# min_pixels = 256*28*28
# max_pixels = 1280*28*28
# processor = AutoProcessor.from_pretrained("Qwen/Qwen2.5-VL-7B-Instruct", min_pixels=min_pixels, max_pixels=max_pixels)


# data

In [ ]:
import pandas as pd
from PIL import Image
from tqdm.auto import tqdm

In [ ]:
data_dir = '../data/'
synthetic_data_dir = '../synthetic_data/'
os.listdir(data_dir)

In [ ]:
disease_images_df = pd.read_csv(os.path.join(data_dir, 'disease_images.csv'))
disease_images_df.head()

In [ ]:
os.listdir(synthetic_data_dir)

In [ ]:
row = disease_images_df.iloc[0]
image_path = os.path.join(data_dir, 'rd_images',row['disease_abbr'], row['image_name'])
image_path

In [ ]:
def extract_assistant_part(text):
    if "assistant" in text:
        return text.split("assistant", 2)[2].strip()
    return text.strip()

In [ ]:
output_dir = os.path.join('../reports/new/', 'rd_images')

for i, row in disease_images_df.iterrows():
    image_path = os.path.join(data_dir, 'rd_images',row['disease_abbr'], row['image_name'])
    image_name = row['image_name']
    suffix = image_name.split('.')[-1]
    output_file_name = image_name.replace(suffix, 'txt')
    disease_name = row['disease_name']
    sub_category = row['sub_category']
    gene_name = row['gene_name']
    orphanet_id = row['orphanet_code']
    text = (
        "You are a professional clinical geneticist.\n"
        "Given the facial image, generate a concise diagnostic report that strictly describes visible facial features only. "
        "**Instructions:**\n"
        "- Begin with an \"Overall Impression\" paragraph summarizing the patient's facial appearance.\n"
        "- Then, for each of the following facial regions, write a single paragraph that describes only what is visibly observed, without suggesting any diagnosis:\n"
        "   - Left Eye\n"
        "   - Right Eye\n"
        "   - Nose\n"
        "   - Mouth/Lips\n"
        "- After all regions, provide a final section \"Prediction of Potential Disease\" where you may synthesize the observations and suggest possible diagnoses.\n"
        "- Conclude with a \"Diagnostic Suggestion\" for next clinical steps.\n"
        "**Facial Diagnostic Report:**\n"
        "Overall Impression:\n"
        "Left Eye:\n"
        "Right Eye:\n"
        "Nose:\n"
        "Mouth/Lips:\n"
        "Prediction of Potential Disease:\n"
        "Diagnostic Suggestion:\n"

    )
    messages = [
        {
            "role": "user",
            "content": [
                {
                    "type": "image",
                    "image": image_path, 
                },
                {
                    "type": "text",
                    "text":  text,
                }
            ]
        }
    ]

    with torch.no_grad():
        # Preparation for inference
        text = processor.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True
        )
        image_inputs, video_inputs = process_vision_info(messages)
        inputs = processor(
            text=[text],
            images=image_inputs,
            videos=video_inputs,
            padding=True,
            return_tensors="pt",
        )
        inputs = inputs.to("cuda")
        
        # Inference: Generation of the output
        generated_ids = model.generate(**inputs, max_new_tokens=1024)
        generated_ids_trimmed = [
            out_ids[len(in_ids) :] for in_ids, out_ids in zip(inputs.input_ids, generated_ids)
        ]

        output_text = processor.batch_decode(
            generated_ids, 
            # generated_ids_trimmed
            skip_special_tokens=True, clean_up_tokenization_spaces=False
        )
        answer = output_text[0]
        with open(os.path.join(output_dir, output_file_name), 'w') as f:
            f.write(extract_assistant_part(answer))

In [ ]:
text = extract_assistant_part(answer)
text

In [ ]:
text.split("assistant", 2)

In [ ]:
with open(os.path.join(output_dir, output_file_name), 'w') as f:
    f.write(answer)

In [ ]:
image_path

In [ ]:
messages = [
    {
        "role": "user",
        "content": [
            {
                "type": "image",
                "image": image_path,  
            },
            {
                "type": "text",
                "text":  text,
            }
        ]
    }
]

In [ ]:
with torch.no_grad():
    # Preparation for inference
    text = processor.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    image_inputs, video_inputs = process_vision_info(messages)
    inputs = processor(
        text=[text],
        images=image_inputs,
        videos=video_inputs,
        padding=True,
        return_tensors="pt",
    )
    inputs = inputs.to("cuda")
    
    # Inference: Generation of the output
    generated_ids = model.generate(**inputs, max_new_tokens=1024)
    generated_ids_trimmed = [
        out_ids[len(in_ids) :] for in_ids, out_ids in zip(inputs.input_ids, generated_ids)
    ]

    output_text = processor.batch_decode(
        generated_ids, 
        # generated_ids_trimmed, 
        skip_special_tokens=True, clean_up_tokenization_spaces=False
    )
    answer = output_text[0]
    print(answer)